# Ноутбук 01: Быстрый старт с HuggingFace Transformers

**Цель:** за 10 минут запустить 5 разных NLP-задач, используя готовые модели.

Никакого обучения. Только inference — берём готовую модель и применяем к тексту.

---
**Датасеты и модели в этом ноутбуке:**
- `distilbert-base-uncased-finetuned-sst-2-english` — анализ тональности
- `Helsinki-NLP/opus-mt-ru-en` — перевод (русский → английский)
- `facebook/bart-large-cnn` — суммаризация
- `dslim/bert-base-NER` — NER
- `distilbert-base-cased-distilled-squad` — вопрос-ответ

In [78]:
# Установка библиотек (если ещё не установлены)
# Раскомментируйте и запустите один раз:
# !pip install transformers datasets torch sentencepiece

In [79]:
# Импортируем главный инструмент — pipeline()
# Pipeline = токенизатор + модель + постобработка в одной функции
from transformers import pipeline

print('Библиотека transformers готова к работе!')

Библиотека transformers готова к работе!


## Задача 1: Анализ тональности текста (Sentiment Analysis)

Задача: определить, какие эмоции содержатся в тексте и степень уверенности модели в правильности детектирования.
Модель: RuBERT, обученный на датасете CEDR.

In [80]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="cointegrated/rubert-tiny2-cedr-emotion-detection",
    top_k=3
)

texts = [
    "Я очень рада, что мой код работает!",
    "Я боюсь что Интернет скоро вообще заблокируют",
    "Я ужасно злюсь из-за этой несправедливости.",
    "Концовка этого фильма опустошает.",
    "Ой, это так неожиданно и приятно!",
    "О, в этот век — преступный и постыдный — Не жить, не чувствовать — удел завидный — Отрадно спать, отрадней — камнем быть."
]

results = emotion_classifier(texts)

emoji_map = {
    "joy": "😄",
    "happiness": "😄",
    "sadness": "😢",
    "anger": "😠",
    "fear": "😨",
    "surprise": "😲",
    "neutral": "😐"
}

for text, emotions in zip(texts, results):
    top3 = []
    for emotion in emotions:
        label = emotion["label"]
        score = emotion["score"]
        emoji = emoji_map.get(label.lower(), "🙂")
        top3.append(f'{emoji} {label.upper()} ({score:.1%})')

    print(f'Текст: "{text}"')
    print("  " + " | ".join(top3))

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2-cedr-emotion-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Текст: "Я очень рада, что мой код работает!"
  😄 JOY (98.3%) | 😲 SURPRISE (1.5%) | 🙂 NO_EMOTION (1.3%)
Текст: "Я боюсь что Интернет скоро вообще заблокируют"
  😨 FEAR (80.4%) | 🙂 NO_EMOTION (16.3%) | 😄 JOY (2.0%)
Текст: "Я ужасно злюсь из-за этой несправедливости."
  😠 ANGER (92.1%) | 😢 SADNESS (6.7%) | 😲 SURPRISE (5.3%)
Текст: "Концовка этого фильма опустошает."
  😢 SADNESS (87.4%) | 😠 ANGER (6.9%) | 🙂 NO_EMOTION (2.5%)
Текст: "Ой, это так неожиданно и приятно!"
  😄 JOY (92.9%) | 😲 SURPRISE (22.1%) | 🙂 NO_EMOTION (1.5%)
Текст: "О, в этот век — преступный и постыдный — Не жить, не чувствовать — удел завидный — Отрадно спать, отрадней — камнем быть."
  😠 ANGER (82.7%) | 😲 SURPRISE (44.7%) | 🙂 NO_EMOTION (4.9%)


## Задача 2: Named Entity Recognition (NER)

Задача: найти в тексте заданные пользователем именованные сущности.
Модель: GLiNER, дообученный на Pile-NER.

In [81]:
#!pip install gliner

In [82]:
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_base")

text = "Здесь поэзия Европы, Азии, Америки — от средневековых сонетов до модернистской лирики XX века. Шекспир, Гёте, Бодлер, Рильке, Лорка, Уитмен — каждый из них говорит о вечных вещах: о любви, ненависти, вере, одиночестве, о том, что значит быть человеком"

labels = ["человек", "тема", "дата", "регион"]

#Тут классификация странная (Бодлер - регион?). Вероятно, модель не знает некоторых поэтов
entities = model.predict_entities(text, labels)

for ent in entities:
    print(ent)

#Заменила "человек" на "фамилия" - стало нормально
labels = ["фамилия", "тема", "дата", "регион"]

entities = model.predict_entities(text, labels)
print("Новые термины")
for ent in entities:
    print(ent)

/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

{'start': 13, 'end': 19, 'text': 'Европы', 'label': 'регион', 'score': 0.9419092535972595}
{'start': 21, 'end': 25, 'text': 'Азии', 'label': 'регион', 'score': 0.9599031209945679}
{'start': 27, 'end': 34, 'text': 'Америки', 'label': 'регион', 'score': 0.9684576988220215}
{'start': 86, 'end': 93, 'text': 'XX века', 'label': 'дата', 'score': 0.9662912487983704}
{'start': 95, 'end': 102, 'text': 'Шекспир', 'label': 'человек', 'score': 0.7147325873374939}
{'start': 104, 'end': 108, 'text': 'Гёте', 'label': 'человек', 'score': 0.6415880918502808}
{'start': 110, 'end': 116, 'text': 'Бодлер', 'label': 'регион', 'score': 0.5236952900886536}
{'start': 118, 'end': 124, 'text': 'Рильке', 'label': 'регион', 'score': 0.6095612645149231}
{'start': 126, 'end': 131, 'text': 'Лорка', 'label': 'регион', 'score': 0.7360377907752991}
{'start': 133, 'end': 139, 'text': 'Уитмен', 'label': 'регион', 'score': 0.7076848745346069}
{'start': 182, 'end': 187, 'text': 'любви', 'label': 'тема', 'score': 0.936634361

## Задача 3: Вопрос-Ответ (Question Answering)

Задача: по тексту-контексту ответить на вопрос.
Модели: русскоязычная RuBERT, дообученная на SberQuAD, многоязычная XLM-RoBERTa Large, дообученная на смеси SQuAD и SberQuAD.

In [83]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

MODELS = {
    "RuBERT_SberQuAD": "MilyaShams/rubert-russian-qa-sberquad",
    "XLMR_multilingual_ru": "AlexKay/xlm-roberta-large-qa-multilingual-finedtuned-ru",
}

context = """
Трансформерная модель была представлена в 2017 году в статье
«Attention Is All You Need» исследователями из Google Brain.
Изначально модель была разработана для машинного перевода.
BERT, GPT и T5 основаны на архитектуре Transformer.
"""

questions = [
    "Когда была представлена трансформерная модель?",
    "Как называется статья, в которой была представлена модель?",
    "Кем была представлена трансформерная модель?",
    "Для чего модель была разработана изначально?",
    "Какие модели основаны на архитектуре Transformer?",
]


def load_model(name: str):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForQuestionAnswering.from_pretrained(name)
    model.eval()
    return tokenizer, model


def answer_question(tokenizer, model, question: str, context: str):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        outputs = model(**inputs)

    start_idx = torch.argmax(outputs.start_logits, dim=1).item()
    end_idx = torch.argmax(outputs.end_logits, dim=1).item()

    if end_idx < start_idx:
        end_idx = start_idx

    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

    start_prob = torch.softmax(outputs.start_logits, dim=-1)[0, start_idx].item()
    end_prob = torch.softmax(outputs.end_logits, dim=-1)[0, end_idx].item()
    score = (start_prob + end_prob) / 2

    return answer, score


loaded_models = {name: load_model(path) for name, path in MODELS.items()}

results = []
for question in questions:
    for model_name, (tokenizer, model) in loaded_models.items():
        answer, score = answer_question(tokenizer, model, question, context)
        results.append({
            "question": question,
            "model": model_name,
            "answer": answer,
            "score": round(score, 4),
        })

df = pd.DataFrame(results)
print(df)

print("\nСравнение по вопросам:\n")
for question in questions:
    print(f"❓ {question}")
    subset = df[df["question"] == question]
    for _, row in subset.iterrows():
        print(f'   [{row["model"]}] {row["answer"]} (score: {row["score"]:.1%})')
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: AlexKay/xlm-roberta-large-qa-multilingual-finedtuned-ru
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/xlm_roberta/modeling_xlm_roberta.py:702: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_bidirectional_mask`. Use `inputs_embeds` instead.
  attention_mask = create_bidirectional_mask(


                                            question                 model  \
0     Когда была представлена трансформерная модель?       RuBERT_SberQuAD   
1     Когда была представлена трансформерная модель?  XLMR_multilingual_ru   
2  Как называется статья, в которой была представ...       RuBERT_SberQuAD   
3  Как называется статья, в которой была представ...  XLMR_multilingual_ru   
4       Кем была представлена трансформерная модель?       RuBERT_SberQuAD   
5       Кем была представлена трансформерная модель?  XLMR_multilingual_ru   
6       Для чего модель была разработана изначально?       RuBERT_SberQuAD   
7       Для чего модель была разработана изначально?  XLMR_multilingual_ru   
8  Какие модели основаны на архитектуре Transformer?       RuBERT_SberQuAD   
9  Какие модели основаны на архитектуре Transformer?  XLMR_multilingual_ru   

                            answer   score  
0                      в 2017 году  0.8003  
1                      в 2017 году  0.9633  
2    «

## Задача 4: Суммаризация текста

Задача: сжать длинный текст в краткое резюме, посмотреть, как выглядят разные резюме при разной максимальной допустимой длине.
Модель: rut5-base, обученная на российских новостных статьях.

In [84]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "IlyaGusev/rut5_base_sum_gazeta"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

article = """
Искусственный интеллект в последние годы достиг значительного прогресса,
особенно в области обработки естественного языка. Большие языковые модели
могут писать статьи, отвечать на сложные вопросы, переводить тексты и
генерировать программный код. Такие модели обучаются на огромных массивах
данных, включающих интернет-страницы, книги, научные статьи и другие источники.
Процесс обучения требует очень больших вычислительных ресурсов и может стоить
миллионы долларов. Несмотря на впечатляющие возможности, современные модели
всё ещё испытывают трудности с некоторыми задачами, например со сложными
математическими рассуждениями, точным планированием на много шагов вперёд
и гарантией фактической достоверности ответов. Исследователи продолжают
работать над тем, чтобы сделать такие системы более эффективными,
точными, безопасными и доступными для практического применения.
"""

inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Набор разных настроек длины
configs = [
    {"min_new_tokens": 15, "max_new_tokens": 30},
    {"min_new_tokens": 25, "max_new_tokens": 50},
    {"min_new_tokens": 40, "max_new_tokens": 80},
    {"min_new_tokens": 60, "max_new_tokens": 120},
]

for cfg in configs:
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            min_new_tokens=cfg["min_new_tokens"],
            max_new_tokens=cfg["max_new_tokens"],
            do_sample=False,
            no_repeat_ngram_size=3
        )

    summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print("=" * 80)
    print(f'min_new_tokens={cfg["min_new_tokens"]}, max_new_tokens={cfg["max_new_tokens"]}')
    print(f'Длина резюме: {len(summary.split())} слов')
    print("Резюме:")
    print(summary)
    print()

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/t5/modeling_t5.py:740: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_bidirectional_mask`. Use `inputs_embeds` instead.
  attention_mask = create_bidirectional_mask(
Both `max_new_tokens` (=30) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/t5/modeling_t5.py:730: FutureWarning: `input_embeds` is depre

min_new_tokens=15, max_new_tokens=30
Длина резюме: 15 слов
Резюме:
Большие языковые модели могут писать статьи, отвечать на сложные вопросы, генерировать программный код и генерировать



Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


min_new_tokens=25, max_new_tokens=50
Длина резюме: 25 слов
Резюме:
Искусственный интеллект в последние годы достиг значительного прогресса в области обработки естественного языка. Исследователи пытаются сделать такие системы более эффективными, точными, безопасными и доступными для



Both `max_new_tokens` (=120) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


min_new_tokens=40, max_new_tokens=80
Длина резюме: 27 слов
Резюме:
Искусственный интеллект в последние годы достиг значительного прогресса в области обработки естественного языка. Исследователи пытаются сделать такие системы более эффективными, точными, безопасными и доступными для практического применения.

min_new_tokens=60, max_new_tokens=120
Длина резюме: 41 слов
Резюме:
Искусственный интеллект в последние годы достиг значительного прогресса в области обработки естественного языка. Исследователи пытаются сделать такие системы более эффективными, точными, безопасными и доступными для практического применения. Большие языковые модели могут писать статьи, отвечать на сложные вопросы и генерировать программный код.



## Задача 5: Генерация текста

Задачи: продолжить заданный текст, заставить gpt2 продолжать текст на русском языке.
Модель для генерации текста: GPT-2.
Модель для перевода: OPUS-MT

In [85]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)
# Генерация текста с GPT-2
GPT2_MODEL = "gpt2"
generator = pipeline('text-generation', model=GPT2_MODEL)

prompt_ru = 'Искусственный интеллект изменит мир путём'

# Генерируем 3 разных варианта продолжения
outputs = generator(
    prompt_ru,
    max_new_tokens=40,    # максимум новых токенов
    num_return_sequences=3,  # сколько вариантов
    temperature=0.9,     # «температура» — разнообразие
    do_sample=True,      # включаем сэмплирование
    pad_token_id=50256   # токен конца строки для GPT-2
)

print(f'Промпт: "{prompt_ru}"')
print()
for i, output in enumerate(outputs, 1):
    text = output['generated_text']
    # Выделяем только новую часть
    new_part = text[len(prompt_ru):]
    print(f'Вариант {i}: ...{new_part}')
    print()

#Для русского промпта gpt2 генерирует бессмыслицу. Попробуем переводить промпты и переводить ответы gpt2 с помощью другой модели.

RU_EN_MODEL = "Helsinki-NLP/opus-mt-ru-en"
EN_RU_MODEL = "Helsinki-NLP/opus-mt-en-ru"

ru_en_tokenizer = AutoTokenizer.from_pretrained(RU_EN_MODEL)
ru_en_model = AutoModelForSeq2SeqLM.from_pretrained(RU_EN_MODEL)

en_ru_tokenizer = AutoTokenizer.from_pretrained(EN_RU_MODEL)
en_ru_model = AutoModelForSeq2SeqLM.from_pretrained(EN_RU_MODEL)

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL)

if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token


def translate(text: str, tokenizer, model) -> str:
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        output_ids = model.generate(**inputs, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

prompt_en = translate(prompt_ru, ru_en_tokenizer, ru_en_model)

inputs = gpt2_tokenizer(prompt_en, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    outputs = generator(
    prompt_en,
    max_new_tokens=40,    # максимум новых токенов
    num_return_sequences=3,  # сколько вариантов
    temperature=0.9,     # «температура» — разнообразие
    do_sample=True,      # включаем сэмплирование
    pad_token_id=50256   # токен конца строки для GPT-2
)

print(f'Русский промпт: "{prompt_ru}"')
print(f'Английский промпт: "{prompt_en}"')
print()

#Смысла все ещё мало, но уже получше!

for i, output in enumerate(outputs, 1):
    text = output['generated_text']
    full_text_ru = translate(text, en_ru_tokenizer, en_ru_model)

    print(f"Вариант {i}")
    print(f"EN full: {text}")
    print(f"RU full: {full_text_ru}")
    print()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Промпт: "Искусственный интеллект изменит мир путём"

Вариант 1: ... Арслове, котореры, С '

Питери Всье пут

Вариант 2: ... на продиталкический празд. Tmukhova reported, with the aid

Вариант 3: ...на.

TOLSTAN'S OCCUPATIONAL RECREATION ON WEST BORD STATIONS – "The following are a list of all available OCCUPATIONAL festivals



/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/marian/modeling_marian.py:565: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_bidirectional_mask`. Use `inputs_embeds` instead.
  attention_mask = create_bidirectional_mask(
/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/marian/modeling_marian.py:761: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(
/home/shion/PycharmProjects/generative_models/.venv/lib/python3.13/site-packages/transformers/models/marian/modeling_marian.py:768: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_bidirectional_mask`. Use `inputs_embeds` instead.
  encoder_attention_mask = create_bidirectional_mask(
Both `max_new_tokens` (=40) and `max_length`(=

Русский промпт: "Искусственный интеллект изменит мир путём"
Английский промпт: "Artificial intelligence will change the world by"

Вариант 1
EN full: Artificial intelligence will change the world by providing an unprecedented level of privacy. And that's what is driving more interest in AI now than ever before—to discover how we can use our own intelligence to improve the lives of others, in particular
RU full: Искусственный интеллект изменит мир, обеспечив беспрецедентный уровень конфиденциальности.

Вариант 2
EN full: Artificial intelligence will change the world by giving us a better understanding of what is going on in our heads and mind.

I don't know whether it's about the AI, but I do know that we have to be a little
RU full: Искусственный интеллект изменит мир, дав нам лучшее представление о том, что происходит в наших головах и мыслях.

Вариант 3
EN full: Artificial intelligence will change the world by getting more people to embrace this technology through advertising and web

## Что произошло «под капотом»?

Каждый вызов `pipeline(...)` делал три вещи:
1. **Токенизация** — текст разбивался на токены и превращался в числа
2. **Прогон через модель** — числа проходили через Transformer
3. **Постобработка** — выходные векторы превращались в понятный результат

В следующих ноутбуках мы разберём каждый шаг подробнее.

In [86]:
# Краткий итог: что мы только что сделали?
tasks = [
    ('Анализ тональности', 'pipeline("sentiment-analysis")',          'POSITIVE/NEGATIVE + score'),
    ('NER',               'pipeline("ner", grouped_entities=True)',    'Список сущностей с метками'),
    ('Вопрос-Ответ',      'pipeline("question-answering")',            'Текст ответа + score'),
    ('Суммаризация',      'pipeline("summarization")',                 'Краткое резюме'),
    ('Генерация',         'pipeline("text-generation")',               'Продолжение текста'),
]

print(f'{"Задача":<22} {"Код":<45} {"Выход"}')
print('-' * 95)
for task, code, output in tasks:
    print(f'{task:<22} {code:<45} {output}')

Задача                 Код                                           Выход
-----------------------------------------------------------------------------------------------
Анализ тональности     pipeline("sentiment-analysis")                POSITIVE/NEGATIVE + score
NER                    pipeline("ner", grouped_entities=True)        Список сущностей с метками
Вопрос-Ответ           pipeline("question-answering")                Текст ответа + score
Суммаризация           pipeline("summarization")                     Краткое резюме
Генерация              pipeline("text-generation")                   Продолжение текста
